In [0]:
retail1_df = spark.read.parquet(
    "abfss://bronze@retaildatasg.dfs.core.windows.net/retail1_details/"
)

display(retail1_df)


In [0]:
retail2_df = spark.read.parquet(
    "abfss://bronze@retaildatasg.dfs.core.windows.net/retail2_details/"
)

display(retail2_df)


In [0]:
product_df = spark.read.parquet(
    "abfss://bronze@retaildatasg.dfs.core.windows.net/product_details/"
)

display(product_df)

Combining both transaction dataset records

In [0]:
trans_df = retail1_df.unionByName(retail2_df)
display(trans_df)

Handling missing values along with all columns which are already in the product dataset

In [0]:
product_df = (
    product_df
    .withColumnRenamed("product_name", "MasterProductName")
    .withColumnRenamed("category", "MasterCategory")
    .withColumnRenamed("price", "MasterPrice")
)

In [0]:
joined_df = trans_df.join(
    product_df,
    "product_id",
    "left"
)

In [0]:
from pyspark.sql.functions import coalesce, col

silver_df = (
    joined_df
    .withColumn(
        "product_name",
        coalesce(col("MasterProductName"), col("product_name"))
    )
    .withColumn(
        "category",
        coalesce(col("MasterCategory"), col("category"))
    )
    .withColumn(
        "price",
        coalesce(col("MasterPrice"), col("price"))
    )
)

silver_df = silver_df.drop(
        "MasterPrice",
        "MasterCategory",
        "MasterProductName"
    )

cols = silver_df.columns

new_cols = ["transaction_id"] + [
    c for c in cols if c != "transaction_id"
]

silver_df = silver_df.select(*new_cols)

display(silver_df)

Remove duplicate records


In [0]:
silver_df = silver_df.dropDuplicates()

Validate Quantities

In [0]:
silver_df.filter(col("quantity") <= 0).show(100)

In [0]:
silver_df = silver_df.filter(col("quantity") > 0)
silver_df = silver_df.orderBy("transaction_id")
display(silver_df)

Checking if all transaction product exist in the product dataset

In [0]:
silver_df.filter(col("product_name").isNull()).show()

In [0]:
from pyspark.sql.functions import col, trim

for c, t in silver_df.dtypes:
    if t == "string":
        silver_df = silver_df.withColumn(c, trim(col(c)))

display(silver_df)

PII masking 

In [0]:
#email masking
from pyspark.sql.functions import col, regexp_replace

silver_df = silver_df.withColumn(
    "email",
    regexp_replace(col("email"), r"(^.).*(@.*$)", r"$1***$2")
)

In [0]:
#phone_number masking
silver_df = silver_df.withColumn(
    "phone",
    regexp_replace(col("phone"), r"(\d{2})\d+(\d{2})", r"$1******$2")
)

Date Column Standardization

In [0]:
from pyspark.sql.functions import col, split, concat_ws, when, to_date

silver_df = silver_df.withColumn(
    "transaction_date_std",
    when(
        split(col("transaction_date"), "-")[1].cast("int") > 12,
        concat_ws(
            "-",
            split(col("transaction_date"), "-")[1],  # day
            split(col("transaction_date"), "-")[0],  # month
            split(col("transaction_date"), "-")[2]   # year
        )
    ).otherwise(col("transaction_date"))
)

In [0]:
silver_df = silver_df.withColumn(
    "transaction_date",
    to_date(col("transaction_date_std"), "dd-MM-yyyy")
).drop("transaction_date_std")

In [0]:
display(silver_df)

Write the silver_df to silver container


In [0]:
silver_df.write \
    .mode("overwrite") \
    .parquet("abfss://silver@retaildatasg.dfs.core.windows.net/trans_details/")

In [0]:
product_df.write \
    .mode("overwrite") \
    .parquet("abfss://silver@retaildatasg.dfs.core.windows.net/product_details/")

In [0]:
silver_df.printSchema()